# Parte 1: Implementación del Stream Cipher

**1.1 Generación del Keystream**

In [23]:
import random


def generar_keystream(seed: str, longitud: int) -> bytes:
    """
    Genera un keystream pseudoaleatorio

    Args:
        seed: Clave para inicializar el PRNG
        longitud: Longitud del keystream en bytes

    Returns:
        Keystream de la longitud especificada
    """
    random.seed(seed)

    # Generar el keystream como bytes aleatorios
    keystream = bytes([random.randint(0, 255) for _ in range(longitud)])

    return keystream

**1.2 Función de Cifrado**

In [24]:
def cifrar(mensaje: str, clave: str) -> bytes:
    """
    Cifra un mensaje usando XOR con keystream

    Args:
        mensaje: Texto plano a cifrar
        clave: Clave para generar el keystream

    Returns:
        Mensaje cifrado
    """
    # Convertir el mensaje a bytes
    mensaje_bytes = mensaje.encode('utf-8')

    # Generar keystream de la misma longitud que el mensaje
    keystream = generar_keystream(clave, len(mensaje_bytes))

    # Aplicar XOR
    texto_cifrado = bytes([m ^ k for m, k in zip(mensaje_bytes, keystream)])

    return texto_cifrado

**1.3 Función de Descifrado**

In [25]:
def descifrar(cifrado: bytes, clave: str) -> str:
    """
    Descifra un mensaje cifrado usando XOR con keystream

    Args:
        cifrado: Mensaje cifrado (bytes)
        clave: Clave para generar el keystream

    Returns:
        Mensaje descifrado (texto plano)
    """
    keystream = generar_keystream(clave, len(cifrado))
    mensaje_bytes = bytes([c ^ k for c, k in zip(cifrado, keystream)])

    try:
        mensaje = mensaje_bytes.decode('utf-8')
    except UnicodeDecodeError:
        mensaje = mensaje_bytes.decode('utf-8', errors='replace')

    return mensaje

**Ejemplo de Uso:**

In [26]:
if __name__ == "__main__":
    mensaje_original = "Javier"
    clave = "Chen"

    print("=" * 60)
    print("STREAM CIPHER")
    print("=" * 60)

    # Cifrar
    print(f"\n1. CIFRADO")
    print("-" * 60)
    cifrado = cifrar(mensaje_original, clave)
    print(f"Mensaje cifrado (hex): {cifrado.hex()}")
    print(f"Mensaje cifrado (bytes): {list(cifrado)}")

    # Descifrar
    print(f"\n2. DESCIFRADO")
    print("-" * 60)
    print(f"Cifrado (hex): {cifrado.hex()}")
    print(f"Clave: '{clave}'")
    descifrado = descifrar(cifrado, clave)
    print(f"Mensaje descifrado: '{descifrado}'")

    # Verificar
    print(f"\n3. VERIFICACIÓN")
    print("-" * 60)
    print(f"Mensaje original:  '{mensaje_original}'")
    print(f"Mensaje descifrado: '{descifrado}'")


STREAM CIPHER

1. CIFRADO
------------------------------------------------------------
Mensaje cifrado (hex): fae802adea6f
Mensaje cifrado (bytes): [250, 232, 2, 173, 234, 111]

2. DESCIFRADO
------------------------------------------------------------
Cifrado (hex): fae802adea6f
Clave: 'Chen'
Mensaje descifrado: 'Javier'

3. VERIFICACIÓN
------------------------------------------------------------
Mensaje original:  'Javier'
Mensaje descifrado: 'Javier'


# Parte 2: Análisis de Seguridad

**2.1 Variación de la Clave**

In [27]:
mensaje_original = "Mensaje secreto importante"
print(f"\nMensaje original: '{mensaje_original}'")

clave_correcta = "clave123"
clave_incorrecta1 = "clave124"
clave_incorrecta2 = "contraseña"

print("\n" + "-" * 70)
print("GENERACIÓN DE KEYSTREAMS CON DIFERENTES CLAVES")
print("-" * 70)

# Generar keystreams
ks_correcto = generar_keystream(clave_correcta, len(mensaje_original))
ks_incorrecto1 = generar_keystream(clave_incorrecta1, len(mensaje_original))
ks_incorrecto2 = generar_keystream(clave_incorrecta2, len(mensaje_original))

print(f"\nClave '{clave_correcta}':")
print(f"  Keystream: {ks_correcto.hex()}")

print(f"\nClave '{clave_incorrecta1}' (1 carácter diferente):")
print(f"  Keystream: {ks_incorrecto1.hex()}")

print(f"\nClave '{clave_incorrecta2}' (completamente diferente):")
print(f"  Keystream: {ks_incorrecto2.hex()}")

print("\n" + "-" * 70)
print("EFECTO EN EL DESCIFRADO")
print("-" * 70)

# Cifrar con clave correcta
cifrado = cifrar(mensaje_original, clave_correcta)
print(f"\nCifrado (hex): {cifrado.hex()}")

# Descifrar con diferentes claves
descifrado_correcto = descifrar(cifrado, clave_correcta)
descifrado_incorrecto1 = descifrar(cifrado, clave_incorrecta1)
descifrado_incorrecto2 = descifrar(cifrado, clave_incorrecta2)

print(f"\nCon clave correcta '{clave_correcta}':")
print(f"  '{descifrado_correcto}' ✓")

print(f"\nCon clave incorrecta '{clave_incorrecta1}':")
print(f"  '{descifrado_incorrecto1}'")

print(f"\nCon clave incorrecta '{clave_incorrecta2}':")
print(f"  '{descifrado_incorrecto2}'")


Mensaje original: 'Mensaje secreto importante'

----------------------------------------------------------------------
GENERACIÓN DE KEYSTREAMS CON DIFERENTES CLAVES
----------------------------------------------------------------------

Clave 'clave123':
  Keystream: bac9cd407aff6c9c332a72674e0e401d29ffebc0028e6c83c9fe

Clave 'clave124' (1 carácter diferente):
  Keystream: 11180f930150d72d1ddf9a18f3422a998b6cd4d6e2114e0da5a0

Clave 'contraseña' (completamente diferente):
  Keystream: 9843786cd6cc0c1cee1dd0e56b359e76506a8e6d1253cb4ba30c

----------------------------------------------------------------------
EFECTO EN EL DESCIFRADO
----------------------------------------------------------------------

Cifrado (hex): f7aca3331b9509bc404f11152b7a2f3d40929baf70fa0dedbd9b

Con clave correcta 'clave123':
  'Mensaje secreto importante' ✓

Con clave incorrecta 'clave124':
�8���Oy��C�;'

Con clave incorrecta 'contraseña':
  'o��_�Y��R��@O�K��b�Ʀ�'


**2.2 Reutilización del Keystream**


In [33]:
def xor_bytes(b1: bytes, b2: bytes) -> bytes:
    return bytes([x ^ y for x, y in zip(b1, b2)])

mensaje1 = "Transferencia Realizada"
mensaje2 = "Transferencia Negada"

clave = "misma_clave"

# Cifrado con la MISMA clave
cipher1 = cifrar(mensaje1, clave)
cipher2 = cifrar(mensaje2, clave)

print("Mensaje 1:", mensaje1)
print("Mensaje 2:", mensaje2)
print("\nTextoCifrado 1:", cipher1.hex())
print("TextoCifrado 2:", cipher2.hex())

# Simulación de atacante
print("\n--- Simulación de atacante ---")

# El atacante intercepta C1 y C2
c1_xor_c2 = xor_bytes(cipher1, cipher2)

print("C1 XOR C2 (hex):", c1_xor_c2.hex())

# Supongamos que el atacante conoce mensaje1
mensaje1_bytes = mensaje1.encode()

# Puede recuperar mensaje2 sin conocer la clave
mensaje2_recuperado = xor_bytes(c1_xor_c2, mensaje1_bytes)

print("\nMensaje 2 recuperado por el atacante:")
print(mensaje2_recuperado.decode())


Mensaje 1: Transferencia Realizada
Mensaje 2: Transferencia Negada

TextoCifrado 1: 037fa3b7b980d6ce718c1b8f602ff0538788dbebaea77b
TextoCifrado 2: 037fa3b7b980d6ce718c1b8f602fec538185d6f0

--- Simulación de atacante ---
C1 XOR C2 (hex): 00000000000000000000000000001c00060d0d1b

Mensaje 2 recuperado por el atacante:
Transferencia Negada


**2.3 Longitud del Keystream**

In [36]:
mensaje = "Mensaje secreto de prueba"
clave = "mi_clave"
mensaje_bytes = mensaje.encode('utf-8')

print(f"\nMensaje: '{mensaje}' ({len(mensaje_bytes)} bytes)")

print("\n" + "-" * 70)
print("CASO 1: KEYSTREAM MÁS CORTO")
print("-" * 70)

longitud_corta = 10
keystream_corto = generar_keystream(clave, longitud_corta)

cifrado_parcial = bytes([mensaje_bytes[i] ^ keystream_corto[i] for i in range(longitud_corta)])
resto_sin_cifrar = mensaje_bytes[longitud_corta:]

print(f"Keystream: {longitud_corta} bytes")
print(f"Parte cifrada: {cifrado_parcial.hex()}")
print(f"Parte SIN CIFRAR: '{resto_sin_cifrar.decode('utf-8')}'")
print(f"  {len(resto_sin_cifrar)} bytes EXPUESTOS")

print("\n" + "-" * 70)
print("CASO 2: KEYSTREAM IGUAL AL MENSAJE")
print("-" * 70)

cifrado_completo = cifrar(mensaje, clave)
descifrado_completo = descifrar(cifrado_completo, clave)

print(f"Keystream: {len(mensaje_bytes)} bytes")
print(f"Cifrado: {cifrado_completo.hex()}")
print(f"Descifrado: '{descifrado_completo}'")
print(f"Todo protegido")

print("\n" + "-" * 70)
print("CASO 3: KEYSTREAM MÁS LARGO")
print("-" * 70)

longitud_larga = len(mensaje_bytes) + 15
keystream_largo = generar_keystream(clave, longitud_larga)

cifrado_largo = bytes([mensaje_bytes[i] ^ keystream_largo[i] for i in range(len(mensaje_bytes))])
bytes_no_usados = longitud_larga - len(mensaje_bytes)

print(f"Keystream: {longitud_larga} bytes")
print(f"Mensaje: {len(mensaje_bytes)} bytes")
print(f"Cifrado: {cifrado_largo.hex()}")
print(f"Bytes desperdiciados: {bytes_no_usados}")
print(f"Seguro pero ineficiente")



Mensaje: 'Mensaje secreto de prueba' (25 bytes)

----------------------------------------------------------------------
CASO 1: KEYSTREAM MÁS CORTO
----------------------------------------------------------------------
Keystream: 10 bytes
Parte cifrada: aff7ef739f557132b466
Parte SIN CIFRAR: 'creto de prueba'
  15 bytes EXPUESTOS

----------------------------------------------------------------------
CASO 2: KEYSTREAM IGUAL AL MENSAJE
----------------------------------------------------------------------
Keystream: 25 bytes
Cifrado: aff7ef739f557132b46683bdcc05d2366638e8344a317009e1
Descifrado: 'Mensaje secreto de prueba'
Todo protegido

----------------------------------------------------------------------
CASO 3: KEYSTREAM MÁS LARGO
----------------------------------------------------------------------
Keystream: 40 bytes
Mensaje: 25 bytes
Cifrado: aff7ef739f557132b46683bdcc05d2366638e8344a317009e1
Bytes desperdiciados: 15
Seguro pero ineficiente


# Parte 3: Validación y Pruebas

**3.1 Ejemplos de Entrada/Salida**

In [51]:
import base64
ejemplos = [
    {
        "nombre": "Ejemplo 1:",
        "mensaje": "Stream-Cipher",
        "clave": "Tarea"
    },
    {
        "nombre": "Ejemplo 2:",
        "mensaje": "Javier Chen",
        "clave": "22153"
    },
    {
        "nombre": "Ejemplo 3:",
        "mensaje": "Cifrado de Información",
        "clave": "Sección10"
    }
]

for i, ejemplo in enumerate(ejemplos, 1):
    print(f"\n{'=' * 70}")
    print(f"{ejemplo['nombre']}")
    print(f"{'=' * 70}")

    mensaje = ejemplo['mensaje']
    clave = ejemplo['clave']

    # Cifrar
    cifrado = cifrar(mensaje, clave)

    # Descifrar
    descifrado = descifrar(cifrado, clave)

    # Mostrar resultados
    print(f"\nTexto plano original:")
    print(f"  '{mensaje}'")

    print(f"\nClave utilizada:")
    print(f"  '{clave}'")

    print(f"\nTexto cifrado (hexadecimal):")
    print(f"  {cifrado.hex()}")

    print(f"\nTexto cifrado (base64):")
    print(f"  {base64.b64encode(cifrado).decode('utf-8')}")

    print(f"\nTexto descifrado:")
    print(f"  '{descifrado}'")


Ejemplo 1:

Texto plano original:
  'Stream-Cipher'

Clave utilizada:
  'Tarea'

Texto cifrado (hexadecimal):
  e1bedcb354e855363bc8000b0a

Texto cifrado (base64):
  4b7cs1ToVTY7yAALCg==

Texto descifrado:
  'Stream-Cipher'

Ejemplo 2:

Texto plano original:
  'Javier Chen'

Clave utilizada:
  '22153'

Texto cifrado (hexadecimal):
  b7c190d844e55634fdde17

Texto cifrado (base64):
  t8GQ2ETlVjT93hc=

Texto descifrado:
  'Javier Chen'

Ejemplo 3:

Texto plano original:
  'Cifrado de Información'

Clave utilizada:
  'Sección10'

Texto cifrado (hexadecimal):
  1bdd6f99d0f50f19b59964636887ca7861f09cd100e914

Texto cifrado (base64):
  G91vmdD1Dxm1mWRjaIfKeGHwnNEA6RQ=

Texto descifrado:
  'Cifrado de Información'


**3.2 Pruebas Unitarias**

In [50]:
print("\n" + "-" * 70)
print("PRUEBA 1: Descifrado recupera exactamente el mensaje original")
print("-" * 70)

mensajes_prueba = [
    "Hola Mundo",
    "Mensaje con ñ y acentuación",
    "1234567890",
    "!@#$%^&*()",
    "Un mensaje más largo para probar que el descifrado funciona correctamente"
]

clave_prueba = "clave_test"
todos_correctos = True

for mensaje in mensajes_prueba:
    cifrado = cifrar(mensaje, clave_prueba)
    descifrado = descifrar(cifrado, clave_prueba)
    correcto = mensaje == descifrado
    todos_correctos = todos_correctos and correcto
    print(f"Mensaje: '{mensaje[:30]}...' → {(' PASS' if correcto else ' FAIL')}")

print("\n" + "-" * 70)
print("PRUEBA 2: Diferentes claves producen diferentes textos cifrados")
print("-" * 70)

mensaje = "Mensaje de prueba"
claves = ["clave1", "clave2", "clave3", "clave4"]
cifrados = []

for clave in claves:
    cifrado = cifrar(mensaje, clave)
    cifrados.append(cifrado)
    print(f"Clave '{clave}': {cifrado.hex()}")

# Verificar que todos sean diferentes
diferentes = len(cifrados) == len(set(cifrados))

print("\n" + "-" * 70)
print("PRUEBA 3: Misma clave produce mismo texto cifrado (determinismo)")
print("-" * 70)

mensaje = "Mensaje determinista"
clave = "clave_fija"
num_pruebas = 5

print(f"Cifrando '{mensaje}' {num_pruebas} veces con la misma clave:\n")

cifrados_determinismo = []
for i in range(num_pruebas):
    cifrado = cifrar(mensaje, clave)
    cifrados_determinismo.append(cifrado)
    print(f"Intento {i+1}: {cifrado.hex()}")

# Verificar que todos sean iguales
todos_iguales = all(c == cifrados_determinismo[0] for c in cifrados_determinismo)

print("\n" + "-" * 70)
print("PRUEBA 4: Cifrado maneja diferentes longitudes correctamente")
print("-" * 70)

mensajes_longitudes = [
    ("Corto", "Hi"),
    ("Medio", "Este es un mensaje mediano"),
    ("Largo", "Este es un mensaje mucho más largo que los anteriores"),
    ("Muy largo", "A" * 200)
]

clave = "clave_longitud"
todos_pasan = True

print(f"{'Caso':<15} {'Long. Original':<15} {'Long. Cifrado':<15} {'¿Descifrado OK?':<20}")
print("-" * 70)

for nombre, mensaje in mensajes_longitudes:
    long_original = len(mensaje.encode('utf-8'))
    cifrado = cifrar(mensaje, clave)
    long_cifrado = len(cifrado)
    descifrado = descifrar(cifrado, clave)

    correcto = mensaje == descifrado
    longitudes_iguales = long_original == long_cifrado

    todos_pasan = todos_pasan and correcto and longitudes_iguales

    print(f"{nombre:<15} {long_original:<15} {long_cifrado:<15} {(' PASS' if correcto else ' FAIL'):<20}")



----------------------------------------------------------------------
PRUEBA 1: Descifrado recupera exactamente el mensaje original
----------------------------------------------------------------------
Mensaje: 'Hola Mundo...' →  PASS
Mensaje: 'Mensaje con ñ y acentuación...' →  PASS
Mensaje: '1234567890...' →  PASS
Mensaje: '!@#$%^&*()...' →  PASS
Mensaje: 'Un mensaje más largo para prob...' →  PASS

----------------------------------------------------------------------
PRUEBA 2: Diferentes claves producen diferentes textos cifrados
----------------------------------------------------------------------
Clave 'clave1': 2ecdd2b04d66858a9021d771cc2b60b5a7
Clave 'clave2': 4b78b1470b52500fe4a34e4e05b72b6d74
Clave 'clave3': ad7619b12d07b05aa93d1bf819564d1ba3
Clave 'clave4': 0cdd7c5c105ef0da2c02e81dd2351179d7

----------------------------------------------------------------------
PRUEBA 3: Misma clave produce mismo texto cifrado (determinismo)
---------------------------------------------